# 第二节 图像分析法实现手写数字二分类

## 实验目标
通过本案例的学习：

1. 掌握使用传统的软件编程方法，而不是AI的方法来实现手写数字识别；
2. 掌握分析图像统计特征的方法；


## 注意事项

1. 本案例推荐使用Pytorch-1.0.0、CPU运行；

2. 如果您是第一次使用 JupyterLab，请查看[《ModelArts JupyterLab使用指导》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0012.html)了解使用方法；

3. 如果您在使用 JupyterLab 过程中碰到报错，请参考[《ModelArts JupyterLab常见问题解决办法》](https://support.huaweicloud.com/modelarts_faq/modelarts_05_0185.html)尝试解决问题。

## 实验步骤

### 案例内容介绍
手写数字识别任务，是要对每张28\*28大小的图片进行预测，判断该图片是数字0-9中的哪一个，因此这是一个10分类的任务。  
做科研的常规方法是先对一个问题做一些假设或简化，尝试去解决这个简单的问题，等简单问题得到较好的解决之后，再减少假设，尝试解决更贴近现实情况、也更复杂的问题。  
本课程也将遵循这种方法，先假设手写数字识别任务只需要识别0和1两个数字，我们先尝试解决这个简单的二分类问题，之后再解决10分类的问题。  
接下来的第2~4节内容都是解决手写数字0和1的二分类问题。  
实现手写数字0和1的二分类，有很多种方法，我们先采用非机器学习的方法，比如基于图像分析的传统编程方法来实现数字0和1的二分类。

### 1. 准备手写数字0和1的数据集
由于整个MNIST数据集是包含0~9的所有图片，我们现在研究的是简化的0和1的二分类问题，所以先从整个数据集中将所有手写数字0和1的图片挑选出来，同样也需要区分训练集和测试集。

In [2]:
import os
import numpy as np
import torchvision.datasets.mnist as mnist

datasets_dir = '../datasets'
if not os.path.exists(datasets_dir):
    os.makedirs(datasets_dir)
if not os.path.exists(os.path.join(datasets_dir, 'MNIST_data.zip')):
    import moxing as mox
    mox.file.copy('obs://modelarts-labs-bj4/course/hwc_edu/deep_learning/datasets/MNIST_data.zip', 
                  os.path.join(datasets_dir, 'MNIST_data.zip'))
    os.system('cd %s; unzip MNIST_data.zip' % (datasets_dir))

# 读取完整训练样本
train_data = mnist.read_image_file(os.path.join(datasets_dir, 'MNIST_data/raw/train-images-idx3-ubyte')).numpy().astype(np.uint8)
train_label = mnist.read_label_file(os.path.join(datasets_dir, 'MNIST_data/raw/train-labels-idx1-ubyte')).numpy().astype(np.uint8)
# 读取完整测试样本
test_data = mnist.read_image_file(os.path.join(datasets_dir, 'MNIST_data/raw/t10k-images-idx3-ubyte')).numpy().astype(np.uint8)
test_label = mnist.read_label_file(os.path.join(datasets_dir, 'MNIST_data/raw/t10k-labels-idx1-ubyte')).numpy().astype(np.uint8)
train_zeros = train_data[train_label == 0]
train_ones = train_data[train_label == 1]
test_zeros = test_data[test_label == 0]
test_ones = test_data[test_label == 1]

print('数字0，训练集规模：', len(train_zeros), '，测试集规模：', len(test_zeros))
print('数字1，训练集规模：', len(train_ones), '，测试集规模：', len(test_ones))

数字0，训练集规模： 5923 ，测试集规模： 980
数字1，训练集规模： 6742 ，测试集规模： 1135


### 2. 进行样本分析

#### 2.1 查看样本的整体概况

In [2]:
# 查看30张数字0的图片
import numpy as np
from PIL import Image
Image.fromarray(np.hstack(train_zeros[:30]))

In [3]:
# 查看30张数字1的图片
Image.fromarray(np.hstack(train_ones[:30]))

#### 2.2 查看单张图片的细节

上一节已经讲到，MNIST数据集中的每张图片都是28\*28大小，使用python模块读取图片文件后，图片可以用一个28\*28的矩阵来表示，下面我们就来查看一下这个矩阵中的具体数值

In [4]:
# 查看图片的像素值
import pandas as pd
df = pd.DataFrame(train_data[1])
df.style.set_properties(**{'font-size':'6pt'}).background_gradient('Greys')

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,51,159,253,159,50,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,48,238,252,252,252,237,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,54,227,253,252,239,233,252,57,6,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,10,60,224,252,253,252,202,84,252,253,122,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,163,252,252,252,253,252,252,96,189,253,167,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,51,238,253,253,190,114,253,228,47,79,255,168,0,0,0,0,0,0


在手写数字原图中，图片的背景是黑色，对应的像素值是0，数字笔画的颜色是白色，对应像素值是255，为了方便显示，上图显示的颜色做了黑白翻转。  
**可以发现一个现象：** 矩阵中的每一个值都代表图片中的一个像素，没有笔画的地方是0像素值，有笔画的地方是非零像素，而且按照常理，同样大小的图片中，数字0的笔画面积一般会比数字1的笔画面积要多  
**由此产生一个思路：** 能否根据笔画产生的非零像素在整幅图像中的占比来区分数字0和1？  
先分别统计数字0和数字1的非零像素在整幅图像中的占比均值，由于数字0的非零像素占比一般比数字1的要大，所以只需要找到一个合适的非零像素占比阈值（用变量th表示），如果某张图片的非零像素占比大于th，就可以将该图片分类为0，否则分类为1。为实现这个思路，我们接下来可以采用传统的编程方法来一步步实现。

### 3. 定义非零像素占比函数 

In [5]:
def calc_nonzero_ratio(img):
    '''实现方法：使用np.count_nonzero函数统计矩阵中的非零像素个数，除以图像大小，即可得到非零像素占比'''
    img = np.asarray(img)
    return np.count_nonzero(img) / img.size

统计数字0的非零像素占比均值

In [6]:
zeros_ratio = 0
for zero in train_zeros:
    zeros_ratio += calc_nonzero_ratio(zero)
zeros_ratio = zeros_ratio / len(train_zeros)
print('数字0的非零像素占比均值：', zeros_ratio)

数字0的非零像素占比均值： 0.24486587223104606


统计数字1的非零像素占比均值

In [7]:
ones_ratio = 0
for one in train_ones:
    ones_ratio += calc_nonzero_ratio(one)
ones_ratio = ones_ratio / len(train_ones)
print('数字1的非零像素占比均值：', ones_ratio)

数字1的非零像素占比均值： 0.10949749968216262


### 4. 设置像素占比分类阈值
先采取一个简单的策略来设置分类阈值，直接取数字0和数字1的非零像素占比的平均值，取4位有效小数

In [8]:
th = round((zeros_ratio + ones_ratio) / 2, 4)
print('分类阈值：', th)

分类阈值： 0.1772


### 5. 定义分类预测函数
这个分类方法很简单，如果某张图片的非零像素占比大于th，就将该图片分类为0，否则分类为1

In [9]:
def predict(img):
    if calc_nonzero_ratio(img) > th:
        pred_label = 0
    else:
        pred_label = 1
    return pred_label

### 6. 准确率统计
对数字0的测试样本进行预测，并统计准确率

In [10]:
zero_right_count = 0
for zero in test_zeros:
    pred_result = predict(zero)
    if pred_result == 0:
        zero_right_count += 1
print('数字0测试样本准确率：%.4f' % (float(zero_right_count) / len(test_zeros)))

数字0测试样本准确率：0.9571


对数字1的测试样本进行预测，并统计准确率

In [11]:
one_right_count = 0
for one in test_ones:
    pred_result = predict(one)
    if pred_result == 1:
        one_right_count += 1
print('数字1测试样本准确率：%.4f' % (float(one_right_count) / len(test_ones)))

数字1测试样本准确率：0.9762


统计综合准确率

In [12]:
print('测试样本综合准确率：%.4f' % (float(zero_right_count + one_right_count) / (len(test_zeros) + len(test_ones))))

测试样本综合准确率：0.9674


如上所示，使用“统计非零像素占比，比较阈值” 这种很简单的策略，也可以实现手写数字0和1的分类，数字0和数字1的分类准确率分别是 95.71% 和 97.62%，综合准确率达到 96.74%

恭喜你，实现了一个手写数字0和1分类效果还不错的模型！